In [ ]:
!pip install -q numpy pillow matplotlib scikit-learn

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:

import io
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from matplotlib.patches import Rectangle
from sklearn.cluster import KMeans

# Convertir RGB a HEX
def rgb_to_hex(color):
    color = np.clip(np.rint(color), 0, 255).astype(int)
    return "#{:02X}{:02X}{:02X}".format(*color)

# Mostrar visualmente una paleta
def mostrar_paleta(colors, percentages, title):
    fig, ax = plt.subplots(figsize=(12, 2))

    for i, (color, percentage) in enumerate(
        zip(colors, percentages)
    ):
        ax.add_patch(
            Rectangle(
                (i, 0), 1, 1,
                color=np.array(color) / 255
            )
        )

        ax.text(
            i + 0.5, -0.12,
            f"{rgb_to_hex(color)}\n{percentage*100:.1f}%",
            ha="center",
            va="top",
            fontsize=9
        )

    ax.set_xlim(0, len(colors))
    ax.set_ylim(-0.55, 1)
    ax.axis("off")
    ax.set_title(title)
    plt.show()


# Guardaremos los colores de las cinco imágenes
todos_los_colores = []
todos_los_pesos = []

numero_imagenes = len(uploaded)

# Procesar cada imagen
for nombre, contenido in uploaded.items():

    imagen = Image.open(
        io.BytesIO(contenido)
    ).convert("RGB")

    # Reducir tamaño para acelerar el cálculo
    imagen.thumbnail((250, 250))

    pixeles = np.array(imagen).reshape(-1, 3)

    # K-Means con 8 grupos
    modelo = KMeans(
        n_clusters=8,
        random_state=42,
        n_init=10
    )

    etiquetas = modelo.fit_predict(pixeles)

    # Calcular proporción de cada color
    cantidades = np.bincount(
        etiquetas,
        minlength=8
    )

    porcentajes = cantidades / cantidades.sum()

    # Ordenar de mayor a menor frecuencia
    orden = np.argsort(porcentajes)[::-1]

    colores = modelo.cluster_centers_[orden]
    porcentajes = porcentajes[orden]

    print("\nIMAGEN:", nombre)

    for i, (color, porcentaje) in enumerate(
        zip(colores, porcentajes), 1
    ):
        print(
            f"Color {i}: {rgb_to_hex(color)} "
            f"- {porcentaje*100:.2f}%"
        )

    mostrar_paleta(
        colores,
        porcentajes,
        f"8 colores dominantes: {nombre}"
    )

    # Guardar los 8 colores y su peso
    todos_los_colores.extend(colores)

    # Cada imagen tiene el mismo peso total
    todos_los_pesos.extend(
        porcentajes / numero_imagenes
    )

print("\nExtracción terminada.")
print("Total de colores obtenidos:", len(todos_los_colores))


In [ ]:

# Convertir los datos en arrays
colores = np.array(todos_los_colores)
pesos = np.array(todos_los_pesos)

# Segundo K-Means: agrupar los 40 colores en 3 grupos
modelo_global = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=20
)

# Cada color pesa según su presencia en las imágenes
modelo_global.fit(
    colores,
    sample_weight=pesos
)

# Obtener los 3 colores representativos
colores_finales = modelo_global.cluster_centers_

# Calcular la presencia de cada grupo
etiquetas = modelo_global.labels_

porcentajes_finales = np.array([
    pesos[etiquetas == i].sum()
    for i in range(3)
])

# Ordenar por importancia
orden = np.argsort(porcentajes_finales)[::-1]

colores_finales = colores_finales[orden]
porcentajes_finales = porcentajes_finales[orden]

print("PALETA CORPORATIVA DE SCOUT\n")

for i, (color, porcentaje) in enumerate(
    zip(colores_finales, porcentajes_finales), 1
):
    print(
        f"Color {i}: {rgb_to_hex(color)} "
        f"- Presencia: {porcentaje*100:.2f}%"
    )

# Mostrar resultado
mostrar_paleta(
    colores_finales,
    porcentajes_finales,
    "Paleta corporativa: 3 colores principales"
)
